# 🎬 CineAI: Professional Video Animation Generator
## USP: AI-Powered Cinematic Storytelling with Film School Mastery

**Key Features:**
- 🎥 Professional cinematography guidelines (angles, lighting, composition)
- 🎬 Film school-grade scene direction
- 🎨 Consistent visual style enforcement
- 🔊 Integrated sound design specifications
- ⚡ Gemini-powered intelligent scene generation

In [5]:
import os
import google.generativeai as genai
import json
from typing import Dict, List
from dataclasses import dataclass

# Configure Gemini API from environment (avoid hardcoded keys)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    print("✅ Gemini API configured from environment variable.")
else:
    print("⚠️ GEMINI_API_KEY not found. Set it in your environment to enable generation.")

⚠️ GEMINI_API_KEY not found. Set it in your environment to enable generation.


In [6]:
# Check available models (only if API key is configured)
print("Available Gemini Models:")
print("-" * 40)
if not GEMINI_API_KEY:
    print("⚠️ Skipped model listing: GEMINI_API_KEY is not configured.")
else:
    try:
        found = False
        for m in genai.list_models():
            if 'generateContent' in m.supported_generation_methods:
                print(f"✓ {m.name}")
                found = True
        if not found:
            print("⚠️ No generateContent-capable models returned for this key/project.")
    except Exception as e:
        print(f"⚠️ Could not list models: {e}")
        print("Tip: verify your API key status and project permissions.")

Available Gemini Models:
----------------------------------------
⚠️ Skipped model listing: GEMINI_API_KEY is not configured.


In [7]:
@dataclass
class CinematicStandards:
    """USP: Consistency enforcement through professional film standards"""
    
    CAMERA_ANGLES = {
        "establishing": "Wide shot, high angle, reveals location context",
        "intimate": "Close-up, eye level, emotional connection",
        "dramatic": "Low angle, dynamic composition, power/tension",
        "voyeuristic": "Over-shoulder, medium shot, third-person perspective",
        "cinematic": "Dutch angle, creative framing, visual interest"
    }
    
    LIGHTING_SCHEMES = {
        "golden_hour": "Warm, soft, 3-point lighting, golden tones",
        "noir": "High contrast, hard shadows, dramatic chiaroscuro",
        "natural": "Ambient, diffused, realistic color temperature",
        "neon_cyberpunk": "Colored gels, high saturation, futuristic mood",
        "moonlight": "Cool blue tones, low key, mysterious atmosphere"
    }
    
    CINEMATOGRAPHY_RULES = {
        "rule_of_thirds": "Subject placement at intersection points",
        "leading_lines": "Guide viewer's eye through composition",
        "depth_of_field": "Bokeh background, sharp subject focus",
        "motion": "Smooth camera movements, motivated transitions",
        "color_grading": "Consistent palette, mood-appropriate tones"
    }
    
    SOUND_DESIGN = {
        "ambient": "Environmental sounds, spatial audio",
        "foley": "Realistic movement sounds, enhanced details",
        "sfx": "Impact sounds, emphasis effects",
        "dialogue": "Clear vocals, appropriate reverb"
    }

In [8]:
class FilmSchoolMasterPrompt:
    """USP: Professional-grade system prompt with cinematic expertise"""
    
    MASTER_SYSTEM_PROMPT = """You are a professional Film Director and Cinematographer. Generate detailed cinematic scene specifications.

For each scene, provide:
- Camera: shot type, angle, movement, lens (e.g., 35mm, 50mm)
- Lighting: setup, color temperature (in Kelvin), mood
- Composition: framing rule, subject placement
- Audio: ambient sounds, foley, music/score, sound effects
- Color: grading style, primary colors, saturation level
- Duration: scene length in seconds
- Action: detailed scene description
- Emotion: intended viewer feeling"""

    @staticmethod
    def get_enhanced_prompt(user_request: str, style: str = "cinematic") -> str:
        """Generate enhanced prompt with film school expertise"""
        style_modifiers = {
            "cinematic": "Hollywood blockbuster style, epic scale, high production value",
            "indie": "Intimate, character-driven, naturalistic lighting, handheld camera",
            "noir": "High contrast B&W, venetian blinds shadows, dramatic angles",
            "documentary": "Observational, natural lighting, authentic moments",
            "music_video": "Dynamic cuts, creative angles, rhythm-based editing",
            "horror": "Low-key lighting, unsettling angles, tension-building sound",
        }
        
        return f"""{FilmSchoolMasterPrompt.MASTER_SYSTEM_PROMPT}

Style: {style_modifiers.get(style, style_modifiers['cinematic'])}
Request: {user_request}

Generate a professional cinematic scene specification."""

In [9]:
class CineAI:
    """USP: Integrated cinematic AI video animation generator"""
    
    def __init__(self, api_key: str | None = None):
        self.standards = CinematicStandards()
        self.film_master = FilmSchoolMasterPrompt()
        self.api_key = (api_key or GEMINI_API_KEY or "").strip()
        self.model_name = None
        self.model = None
        
        if not self.api_key:
            print("⚠️ CineAI initialized without API key. Generation methods will return guidance messages.")
            return
        
        try:
            genai.configure(api_key=self.api_key)
            
            # Configure safety settings to prevent blocking
            from google.generativeai.types import HarmCategory, HarmBlockThreshold
            
            safety_settings = {
                HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
                HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
            }
            
            self.model_name = self._select_model_name()
            self.model = genai.GenerativeModel(
                self.model_name,
                safety_settings=safety_settings
            )
            print(f"✅ CineAI ready with model: {self.model_name}")
        except Exception as e:
            self.model = None
            print(f"⚠️ Could not initialize Gemini model: {e}")
    
    def _select_model_name(self) -> str:
        preferred_models = [
            "models/gemini-2.0-flash",
            "models/gemini-1.5-flash-latest",
            "models/gemini-1.5-flash",
        ]
        
        try:
            available = []
            for model in genai.list_models():
                if "generateContent" in model.supported_generation_methods:
                    available.append(model.name)
            
            for candidate in preferred_models:
                if candidate in available:
                    return candidate
            
            if available:
                return available[0]
        except Exception as e:
            print(f"⚠️ Model auto-discovery failed: {e}")
        
        # Fallback for environments where listing fails
        return "models/gemini-2.0-flash"
        
    def generate_animation_spec(
        self, 
        concept: str, 
        style: str = "cinematic",
        num_scenes: int = 3,
        apply_standards: bool = True
    ) -> Dict:
        """
        Generate professional video animation specifications
        
        Args:
            concept: Story/animation concept
            style: Cinematic style (cinematic, indie, noir, etc.)
            num_scenes: Number of scenes to generate
            apply_standards: Apply consistency standards
        
        Returns:
            Complete animation specification with technical details
        """
        
        if self.model is None:
            return {
                "concept": concept,
                "style": style,
                "scenes": num_scenes,
                "specification": "⚠️ Gemini model is unavailable. Set a valid GEMINI_API_KEY and re-run setup cells.",
                "metadata": {
                    "standards_applied": apply_standards,
                    "film_school_prompt": True,
                    "generated_with": "Unavailable"
                }
            }
        
        # Build enhanced prompt with film school mastery
        enhanced_prompt = self.film_master.get_enhanced_prompt(concept, style)
        
        if apply_standards:
            enhanced_prompt += f"\n\nCONSISTENCY STANDARDS:\n"
            enhanced_prompt += f"- Camera Angles: {list(self.standards.CAMERA_ANGLES.keys())}\n"
            enhanced_prompt += f"- Lighting: {list(self.standards.LIGHTING_SCHEMES.keys())}\n"
            enhanced_prompt += f"- Composition: {list(self.standards.CINEMATOGRAPHY_RULES.keys())}\n"
        
        enhanced_prompt += f"\n\nGenerate exactly {num_scenes} scenes with complete specifications."
        
        # Generate with Gemini
        print(f"🎬 Generating {num_scenes} scenes for: {concept[:50]}...")
        
        try:
            response = self.model.generate_content(
                enhanced_prompt,
                generation_config=genai.GenerationConfig(
                    temperature=0.7,
                    max_output_tokens=1500
                )
            )
            
            # Check if response has text
            if response and hasattr(response, 'text') and response.text:
                spec_text = response.text
            elif response and hasattr(response, 'parts'):
                spec_text = "\n".join([part.text for part in response.parts if hasattr(part, 'text')])
            else:
                spec_text = f"⚠️ Response received but no text generated. Safety ratings: {response.prompt_feedback if hasattr(response, 'prompt_feedback') else 'N/A'}"
        except Exception as e:
            spec_text = f"⚠️ Generation failed: {e}"
        
        return {
            "concept": concept,
            "style": style,
            "scenes": num_scenes,
            "specification": spec_text,
            "metadata": {
                "standards_applied": apply_standards,
                "film_school_prompt": True,
                "generated_with": self.model_name or "Unavailable"
            }
        }
    
    def quick_scene(self, description: str, mood: str = "dramatic") -> str:
        """Quick scene generation with automatic professional specs"""
        
        if self.model is None:
            return "⚠️ Gemini model is unavailable. Set a valid GEMINI_API_KEY and re-run setup cells."
        
        prompt = f"""{self.film_master.MASTER_SYSTEM_PROMPT}

Generate ONE complete scene specification for:
DESCRIPTION: {description}
MOOD: {mood}

Apply full cinematic technical details."""
        
        print(f"🎬 Generating scene: {description[:60]}...")
        
        try:
            response = self.model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    temperature=0.7,
                    max_output_tokens=800
                )
            )
            
            # Check if response has text
            if response and hasattr(response, 'text') and response.text:
                return response.text
            if response and hasattr(response, 'parts'):
                return "\n".join([part.text for part in response.parts if hasattr(part, 'text')])
            return f"⚠️ Response received but no text generated.\nPrompt Feedback: {response.prompt_feedback if hasattr(response, 'prompt_feedback') else 'N/A'}\nCandidates: {len(response.candidates) if hasattr(response, 'candidates') else 0}"
        except Exception as e:
            return f"⚠️ Generation failed: {e}"
    
    def get_standards_reference(self) -> Dict:
        """Return all cinematic standards for reference"""
        return {
            "camera_angles": self.standards.CAMERA_ANGLES,
            "lighting_schemes": self.standards.LIGHTING_SCHEMES,
            "cinematography_rules": self.standards.CINEMATOGRAPHY_RULES,
            "sound_design": self.standards.SOUND_DESIGN
        }

In [10]:
# Initialize CineAI
cineai = CineAI()

# Example 1: Quick scene generation
print("=" * 60)
print("EXAMPLE 1: Quick Professional Scene")
print("=" * 60)

quick_scene = cineai.quick_scene(
    description="A lone astronaut discovers an ancient alien artifact on Mars",
    mood="mysterious"
)
print(quick_scene)

⚠️ CineAI initialized without API key. Generation methods will return guidance messages.
EXAMPLE 1: Quick Professional Scene
⚠️ Gemini model is unavailable. Set a valid GEMINI_API_KEY and re-run setup cells.


In [11]:
# Example 2: Full cinematic animation specification
print("\n" + "=" * 60)
print("EXAMPLE 2: Full Multi-Scene Animation Spec")
print("=" * 60)

animation_spec = cineai.generate_animation_spec(
    concept="A cyberpunk detective chases a rogue AI through neon-lit city streets",
    style="noir",
    num_scenes=3,
    apply_standards=True
)

print(f"\nConcept: {animation_spec['concept']}")
print(f"Style: {animation_spec['style']}")
print(f"Scenes: {animation_spec['scenes']}")
print(f"\nMetadata: {json.dumps(animation_spec['metadata'], indent=2)}")
print("\n" + "-" * 60)
print("FULL SPECIFICATION:")
print("-" * 60)
print(animation_spec['specification'])


EXAMPLE 2: Full Multi-Scene Animation Spec

Concept: A cyberpunk detective chases a rogue AI through neon-lit city streets
Style: noir
Scenes: 3

Metadata: {
  "standards_applied": true,
  "film_school_prompt": true,
  "generated_with": "Unavailable"
}

------------------------------------------------------------
FULL SPECIFICATION:
------------------------------------------------------------
⚠️ Gemini model is unavailable. Set a valid GEMINI_API_KEY and re-run setup cells.


In [12]:
# Example 3: View cinematic standards reference
print("\n" + "=" * 60)
print("EXAMPLE 3: Cinematic Standards Reference")
print("=" * 60)

standards = cineai.get_standards_reference()
print("\n📹 CAMERA ANGLES:")
for angle, description in standards['camera_angles'].items():
    print(f"  • {angle}: {description}")

print("\n💡 LIGHTING SCHEMES:")
for scheme, description in standards['lighting_schemes'].items():
    print(f"  • {scheme}: {description}")

print("\n🎬 CINEMATOGRAPHY RULES:")
for rule, description in standards['cinematography_rules'].items():
    print(f"  • {rule}: {description}")

print("\n🔊 SOUND DESIGN:")
for sound, description in standards['sound_design'].items():
    print(f"  • {sound}: {description}")


EXAMPLE 3: Cinematic Standards Reference

📹 CAMERA ANGLES:
  • establishing: Wide shot, high angle, reveals location context
  • intimate: Close-up, eye level, emotional connection
  • dramatic: Low angle, dynamic composition, power/tension
  • voyeuristic: Over-shoulder, medium shot, third-person perspective
  • cinematic: Dutch angle, creative framing, visual interest

💡 LIGHTING SCHEMES:
  • golden_hour: Warm, soft, 3-point lighting, golden tones
  • noir: High contrast, hard shadows, dramatic chiaroscuro
  • natural: Ambient, diffused, realistic color temperature
  • neon_cyberpunk: Colored gels, high saturation, futuristic mood
  • moonlight: Cool blue tones, low key, mysterious atmosphere

🎬 CINEMATOGRAPHY RULES:
  • rule_of_thirds: Subject placement at intersection points
  • leading_lines: Guide viewer's eye through composition
  • depth_of_field: Bokeh background, sharp subject focus
  • motion: Smooth camera movements, motivated transitions
  • color_grading: Consistent pale

## 🎨 Custom Animation Generation

Run the cell below with your own concept!

In [13]:
# 🎬 YOUR CUSTOM ANIMATION
# Modify these parameters and run!

YOUR_CONCEPT = "A magical forest where trees whisper secrets to a young wanderer"
YOUR_STYLE = "cinematic"  # Options: cinematic, indie, noir, documentary, music_video, horror, sci-fi
NUM_SCENES = 3

# Generate your custom animation
custom_animation = cineai.generate_animation_spec(
    concept=YOUR_CONCEPT,
    style=YOUR_STYLE,
    num_scenes=NUM_SCENES,
    apply_standards=True
)

print("🎬 YOUR CUSTOM ANIMATION SPECIFICATION")
print("=" * 80)
print(custom_animation['specification'])

🎬 YOUR CUSTOM ANIMATION SPECIFICATION
⚠️ Gemini model is unavailable. Set a valid GEMINI_API_KEY and re-run setup cells.
